# A mixed fleet

Two aircraft, two different airframes: a DJI M600 multirotor against a small fixed-wing. They are
genuinely different vehicles — one can hover and sidestep, the other must keep flying and bank to
turn — so putting them in the same encounter means each carries its own physics and its own limits.

This notebook shows how to declare that, and runs it under **both** estimators. No plots; the
question here is only how to say it and whether the numbers come out.

In [8]:
import dataclasses

import pandas as pd

from opencdarr import (
    IPS,
    MC,
    MVP,
    Airframe,
    Comm,
    FixedWing,
    Fixed,
    Sweep,
    GnssNavigation,
    LastKnown,
    M600,
    Methods,
    Multirotor,
    PastCPA,
    SMALL_FIXEDWING,
    StateBased,
    run_experiment,
)
from opencdarr.config import load_config

base = load_config("../../configs/pairwise.yaml")

print(f"{'airframe':<18} {'v_min':>7} {'v_max':>7} {'max bank':>9}")
for name, perf in (("M600", M600), ("SMALL_FIXEDWING", SMALL_FIXEDWING)):
    print(f"{name:<18} {perf.v_min:>7.1f} {perf.v_max:>7.1f} {perf.phi_max:>8.0f}°")

airframe             v_min   v_max  max bank
M600                 -18.0    18.0        0°
SMALL_FIXEDWING       12.0    25.0       44°


Read those two rows before going further, because they decide everything below.

The multirotor will fly anywhere from −18 to 18 m/s — backwards included — and has no bank angle at
all, because it does not turn by banking. The fixed-wing **stalls below 12 m/s** and banks up to
44°. There is no single speed that suits both by accident; you have to pick one for each.

## An `Airframe` is the pair that must agree

A `Performance` is what an aircraft can do; a `Kinematics` is how it moves. They are separate
objects, and they have to match — hand the fixed-wing integrator a multirotor's envelope and it sees
`phi_max = 0`, never banks, and therefore never turns. It would fly perfectly straight for the whole
encounter without raising anything.

`Airframe` bundles the two so that mistake cannot be written down:

In [2]:
copter = Airframe(M600, Multirotor())
plane = Airframe(SMALL_FIXEDWING, FixedWing())
for label, af in (("copter", copter), ("plane", plane)):
    print(f"{label:<8} {type(af.kinematics).__name__:<12} "
          f"envelope [{af.perf.v_min:.0f}, {af.perf.v_max:.0f}] m/s, bank {af.perf.phi_max:.0f}°")

try:
    Airframe(M600, FixedWing())          # a fixed-wing on a multirotor envelope
except ValueError as e:
    print(f"\nrefused: {e}")

copter   Multirotor   envelope [-18, 18] m/s, bank 0°
plane    FixedWing    envelope [12, 25] m/s, bank 44°

refused: FixedWing was given an envelope it cannot fly: phi_max must be > 0 (a fixed-wing turns by banking); roll_rate_max must be > 0 (bank could never change); v_min is the stall speed and must be > 0. This looks like a multirotor envelope (e.g. M600); pass a fixed-wing Performance such as SMALL_FIXEDWING.


`Airframe(M600)` on its own is also fine — leaving the kinematics out takes the default
`Multirotor`, so the pair is still consistent.

## Declaring the fleet

`Methods` normally carries **one** `perf` and **one** `kinematics`, which is right when every
aircraft is the same airframe. A mixed fleet says otherwise with `airframes`: one entry per
aircraft, **ownship first**.

In [3]:
FLEETS = {
    "uniform": [Airframe(M600), Airframe(M600)],                 # multirotor v multirotor
    "mixed": [Airframe(M600), Airframe(SMALL_FIXEDWING, FixedWing())],  # multirotor v fixed-wing
}

# everything that is NOT per-aircraft: one detector, one resolver, one recovery, one datalink.
# A mixed fleet is mixed in airframe, not in separation logic.
stack = Methods(
    detector=StateBased(), resolver=MVP(1.05), recovery=PastCPA(),
    navigation=GnssNavigation(), communication=Comm(), surveillance=LastKnown(),
)

try:
    Methods(detector=StateBased(), perf=M600, airframes=FLEETS["mixed"])
except ValueError as e:
    print(f"one spelling at a time: {e}")

one spelling at a time: give either airframes=[...] (one per aircraft) or perf=/kinematics= (one shared airframe), not both


## Each aircraft needs a speed it can actually fly

This is the part that catches people. The sampler sets the ownship's speed with `speed` and the
intruder's with `gs_intr`. Leave `gs_intr` out and the intruder simply copies the ownship — which is
fine when both are the same airframe and wrong the moment they are not.

Below, the ownship (multirotor) cruises at 15 m/s and the fixed-wing intruder at 17 m/s. Both sit
inside their own envelopes. Get it wrong and the run is refused rather than flown:

In [4]:
from opencdarr import AircraftState, Agent

too_slow = AircraftState(id="INT", lat=52.0, lon=4.0, trk=0.0, gs=10.0)   # stall is 12
try:
    Agent(too_slow, SMALL_FIXEDWING, FixedWing())
except ValueError as e:
    print(f"refused: {e}")

refused: initial ground speed 10.0 m/s for 'INT' is outside its envelope [12.0, 25.0] m/s. In a mixed fleet, set each aircraft's speed for its own airframe (the sampler's `speed` / `gs_intr`).


## Running it, on both estimators

The declaration is identical for the two backends — only the `backend=` argument changes. That is
the whole point of the runner: `MC` counts encounters, `IPS` splits toward the rare event, and the
objects they drive are the same ones.

A 90° crossing, head-on (`dcpa = 0`), with a fairly noisy GNSS fix so that losses are frequent
enough for Monte Carlo to see and rare enough to be worth splitting.

In [15]:
rows

[{'fleet': 'uniform',
  'MC P(LoS)': 0.0,
  'MC 95% lo': 0.0,
  'MC 95% hi': 0.43449149475208104},
 {'fleet': 'mixed',
  'MC P(LoS)': 0.0,
  'MC 95% lo': 0.0,
  'MC 95% hi': 0.43449149475208104}]

In [21]:
cfg = dataclasses.replace(
    base, scenario=dataclasses.replace(
        base.scenario, speed=15.0, tlos=180.0, pos_ci95=60.0, vel_ci95=6.0),
)

GEOMETRY = {"dpsi": Sweep([50, 60, 70]), "dcpa": Fixed(0.0), "gs_intr": Fixed(17.0)}

# a decreasing ladder of separations ending at rpz, tuned on the *safer* fleet so its hops
# (the tighter ones) stay healthy; the more dangerous fleet then has slack.
SHELLS = [463, 265, 149, 111, 84, 66, 57, 50]

rows = []
for name, fleet in FLEETS.items():
    declared = {**GEOMETRY, "airframes": Fixed(fleet)}

    for r in run_experiment(declared, ...).records():
        mc = run_experiment(declared, methods=stack, backend=MC(n_encounters=5),
                            base_config=cfg, seed=0, cache=True, n_jobs=4).records()
        # ips = run_experiment(declared, methods=stack,
        #                      backend=IPS(shells=SHELLS, n_particles=300, reps=8),
        #                      base_config=cfg, seed=0, cache=True, n_jobs=4).records()[0]
        rows.append({"fleet": name, "dpsi": r["dpsi"], "MC P(LoS)": r["p_los"], ...})
        
        rows.append({
            "fleet": name,
            "dpsi": r["dpsi"],
            "MC P(LoS)": r["p_los"],
            "MC 95% lo": r["p_los_lo"], "MC 95% hi": r["p_los_hi"],
            # "IPS P(LoS)": ips["p_los"],
            # "IPS 95% lo": ips["p_los_lo"], "IPS 95% hi": ips["p_los_hi"],
            # "collapsed": f"{ips['n_collapsed']}/{ips['reps']}",
        })

pd.DataFrame(rows).set_index("fleet").round(5)

SyntaxError: ':' expected after dictionary key (4019704792.py, line 22)

Two things to read off the table.

**The two estimators agree**, which is the check that matters before trusting either. They sample
completely differently — Monte Carlo runs 3000 whole encounters and counts, IPS concentrates 300
particles through eight shells and multiplies the survival fractions — so agreement is real evidence
rather than a tautology. If the intervals had not overlapped, the shells would be the first suspect.

**The mixed fleet loses separation less often than the uniform one.** Worth being careful about why:
this is not "fixed-wings are safer". The two encounters differ in airframe *and* in the intruder's
speed and turn behaviour, and a faster intruder crossing at 90° spends less time near the ownship.
Attributing the difference would need the speed held constant, which the envelopes do not allow —
the fixed-wing cannot fly at the multirotor's slow end. That is a real limit of comparing airframes,
not a defect in the measurement.

`collapsed` being `0/8` in both rows is the health flag to check on any IPS result: a replication
that finds no survivors at some shell reports `P = 0`, which is not a real zero.

## What is per-aircraft, and what is not

| per aircraft (`Airframe`) | shared by the run (`Methods`) |
|---|---|
| `Performance` — the envelope | detection, resolution, recovery |
| `Kinematics` — how it moves | navigation, communication, surveillance |
| its speed, via `speed` / `gs_intr` | the wind |

So you can fly a multirotor against a fixed-wing, and you cannot give them different resolvers.
That is deliberate: the airframe is a property of the aircraft, while the separation rules and the
datalink are properties of the airspace everyone is operating in.

One thing that comes free and is easy to miss: a fixed-wing cannot fly the raw velocity vector a
resolver produces, so its commands are projected onto course and airspeed on the way to the
airframe. The runner picks that adapter per aircraft from the `Airframe` — the multirotor gets none,
the fixed-wing gets one — so a mixed fleet works without you arranging it.